In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import pickle
from tqdm.auto import tqdm
import pandas as pd
import seaborn as sns
import datetime

# 导入您的自定义数据加载器
# 假设该文件名是brain_voxel_dataloader.py
from brain_voxel_dataloader import BrainVoxelDataLoader

# 定义模型类
class DenseModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=4096, num_classes=102, dropout_rate=0.5):
        super(DenseModel, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer4 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.output_layer = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.output_layer(x)

# 训练模型函数
def train_model(train_loader, val_loader, input_dim=341, num_classes=102, device='cuda', 
                hidden_dim=4096, dropout_rate=0.5, learning_rate=0.00001, weight_decay=0.00001, 
                no_epochs=25):
    # 创建模型
    model = DenseModel(
        input_dim=input_dim, 
        hidden_dim=hidden_dim, 
        num_classes=num_classes, 
        dropout_rate=dropout_rate
    ).to(device)
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(), 
        lr=learning_rate, 
        weight_decay=weight_decay
    )
    
    # 初始化训练历史
    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    # 训练模型
    for epoch in range(no_epochs):
        # 训练阶段
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        all_train_preds = []
        all_train_targets = []
        
        # 进度条
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{no_epochs} [Train]')
        
        for batch in pbar:
            # 处理数据中包含病人ID的情况
            if len(batch) == 3:  # (features, labels, patient_ids)
                inputs, labels, _ = batch
            else:  # (features, labels)
                inputs, labels = batch
            
            # 转换为PyTorch张量，不指定数据类型
            inputs = torch.tensor(inputs, dtype=torch.float32).to(device)
            
            # 处理one-hot标签
            if labels.ndim > 1 and labels.shape[1] > 1:  # one-hot格式
                labels_tensor = torch.tensor(labels)
                target_indices = torch.argmax(labels_tensor, dim=1).to(device)
            else:  # 索引格式
                target_indices = torch.tensor(labels, dtype=torch.long).to(device)
            
            # 前向传播
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, target_indices)
            
            # 反向传播和优化
            loss.backward()
            optimizer.step()
            
            # 统计
            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            train_total += target_indices.size(0)
            train_correct += (predicted == target_indices).sum().item()
            
            # 收集预测和目标用于计算F1
            all_train_preds.extend(predicted.cpu().numpy())
            all_train_targets.extend(target_indices.cpu().numpy())
            
            # 更新进度条
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{train_correct/train_total:.4f}'
            })
        
        train_loss = train_loss / train_total
        train_acc = train_correct / train_total
        train_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
        
        # 验证阶段
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_val_preds = []
        all_val_targets = []
        
        with torch.no_grad():
            # 进度条
            pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{no_epochs} [Val]')
            
            for batch in pbar:
                # 处理数据中包含病人ID的情况
                if len(batch) == 3:  # (features, labels, patient_ids)
                    inputs, labels, _ = batch
                else:  # (features, labels)
                    inputs, labels = batch
                
                # 转换为PyTorch张量，不指定数据类型
                inputs = torch.tensor(inputs, dtype=torch.float32).to(device)
                
                # 处理one-hot标签
                if labels.ndim > 1 and labels.shape[1] > 1:  # one-hot格式
                    labels_tensor = torch.tensor(labels)
                    target_indices = torch.argmax(labels_tensor, dim=1).to(device)
                else:  # 索引格式
                    target_indices = torch.tensor(labels, dtype=torch.long).to(device)
                
                # 前向传播
                outputs = model(inputs)
                loss = criterion(outputs, target_indices)
                
                # 统计
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += target_indices.size(0)
                val_correct += (predicted == target_indices).sum().item()
                
                # 收集预测和目标用于计算F1
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_targets.extend(target_indices.cpu().numpy())
                
                # 更新进度条
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{val_correct/val_total:.4f}'
                })
        
        val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
        
        # 记录历史
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        
        # 打印当前epoch的结果
        print(f"Epoch {epoch+1}/{no_epochs}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    
    return model, history

# 评估模型函数
def evaluate_model(model, test_loader, device='cuda'):
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    all_preds = []
    all_targets = []
    
    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            # 处理数据中包含病人ID的情况
            if len(batch) == 3:  # (features, labels, patient_ids)
                inputs, labels, _ = batch
            else:  # (features, labels)
                inputs, labels = batch
            
            # 转换为PyTorch张量，不指定数据类型
            inputs = torch.tensor(inputs, dtype=torch.float32).to(device)
            
            # 处理one-hot标签
            if labels.ndim > 1 and labels.shape[1] > 1:  # one-hot格式
                labels_tensor = torch.tensor(labels)
                target_indices = torch.argmax(labels_tensor, dim=1).to(device)
            else:  # 索引格式
                target_indices = torch.tensor(labels, dtype=torch.long).to(device)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, target_indices)
            
            # 统计
            test_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            test_total += target_indices.size(0)
            test_correct += (predicted == target_indices).sum().item()
            
            # 收集预测和目标
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target_indices.cpu().numpy())
    
    test_loss = test_loss / test_total
    test_acc = test_correct / test_total
    
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")
    
    # 计算分类报告
    report = classification_report(all_targets, all_preds)
    print("Classification Report:")
    print(report)
    
    return test_loss, test_acc, all_preds, all_targets




In [ ]:

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 设置超参数
batch_size = 128
input_dim = 341
num_classes = 102

# 生成时间戳，用于保存文件
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = f'./output/{timestamp}'
os.makedirs(output_dir, exist_ok=True)

# 初始化数据加载器 - 启用交叉验证
data_loader = BrainVoxelDataLoader(
    base_dir='/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/reorganized_fold_data',  # 修改为实际路径
    cross_validation=True,  # 启用交叉验证
    cv_fold=30,  # 30折交叉验证
    standardize=True,  # 启用标准化
    output_dir=output_dir,  # 使用带时间戳的输出目录
    shuffle=True,  # 打乱训练集
    random_seed=42,  # 随机种子
    save_scaler=True  # 保存标准化器
)

# 获取测试集
test_dataset = data_loader.get_test_dataset()
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# 执行30折交叉验证
cv_results = []
for fold in range(30):
    print(f"\n开始交叉验证第 {fold+1}/30 折")
    
    # 获取当前折的训练集和验证集
    train_dataset, val_dataset = data_loader.get_cv_fold(fold)
    
    print(f"训练集大小: {len(train_dataset)}")
    print(f"验证集大小: {len(val_dataset)}")
    print(f"测试集大小: {len(test_dataset)}")
    
    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 为当前折创建目录
    fold_dir = os.path.join(output_dir, f'fold_{fold+1}')
    os.makedirs(fold_dir, exist_ok=True)
    
    # 训练模型
    model, history = train_model(
        train_loader, 
        val_loader, 
        input_dim=input_dim, 
        num_classes=num_classes, 
        device=device
    )
    
    # 评估模型
    test_loss, test_acc, y_pred, y_true = evaluate_model(model, test_loader, device=device)
    
    # 保存模型和结果
    torch.save(model.state_dict(), os.path.join(fold_dir, 'model.pth'))
    
    # 将当前折的结果添加到交叉验证结果中
    fold_result = {
        'fold': fold + 1,
        'test_loss': test_loss,
        'test_acc': test_acc,
        'history': history
    }
    cv_results.append(fold_result)
    
    # 保存当前折的标准化器（如果可用）
    if hasattr(data_loader, 'scaler') and data_loader.scaler is not None:
        scaler_path = os.path.join(fold_dir, f'scaler_fold_{fold+1}.pkl')
        with open(scaler_path, 'wb') as f:
            pickle.dump(data_loader.scaler, f)
        print(f"标准化器已保存到: {scaler_path}")

# 保存所有交叉验证结果
cv_results_path = os.path.join(output_dir, f'cv_results_{timestamp}.pkl')
with open(cv_results_path, 'wb') as f:
    pickle.dump(cv_results, f)
print(f"交叉验证结果已保存到: {cv_results_path}")

# 计算并显示交叉验证平均结果
mean_test_loss = np.mean([result['test_loss'] for result in cv_results])
mean_test_acc = np.mean([result['test_acc'] for result in cv_results]) * 100  # 转换为百分比
std_test_loss = np.std([result['test_loss'] for result in cv_results])
std_test_acc = np.std([result['test_acc'] for result in cv_results]) * 100  # 转换为百分比

print(f"\n交叉验证平均测试损失: {mean_test_loss:.4f} ± {std_test_loss:.4f}")
print(f"交叉验证平均测试准确率: {mean_test_acc:.2f}% ± {std_test_acc:.2f}%")

# 绘制交叉验证结果
plt.figure(figsize=(10, 6))
plt.errorbar(
    range(1, 31), 
    [result['test_acc'] * 100 for result in cv_results],  # 转换为百分比
    yerr=std_test_acc, 
    fmt='o-', 
    capsize=5
)
plt.axhline(y=mean_test_acc, color='r', linestyle='--', label=f'Mean: {mean_test_acc:.2f}%')
plt.title('Cross-Validation Results: Test Accuracy')
plt.xlabel('Fold')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()
plt.savefig(os.path.join(output_dir, f'cv_results_{timestamp}.png'))
print(f"交叉验证结果图已保存到: {os.path.join(output_dir, f'cv_results_{timestamp}.png')}")